In [1]:
print("Jupyter notebook is working")


Jupyter notebook is working


Part 2: 

In [2]:
%%file producer.py
from kafka import KafkaProducer
import json
import random
import time
from datetime import datetime

# Connect to Kafka
producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

# Data options
stores = ["Warsaw", "Krakow", "Gdansk", "Wroclaw"]
categories = ["electronics", "clothing", "food", "books"]

# Function to generate transaction
def generate_transaction(tx_num):
    return {
        "tx_id": f"TX{tx_num:04d}",
        "user_id": f"u{random.randint(1, 20):02d}",
        "amount": round(random.uniform(5.0, 5000.0), 2),
        "store": random.choice(stores),
        "category": random.choice(categories),
        "timestamp": datetime.now().isoformat()
    }

# Send 50 transactions (1 per second)
for i in range(1, 51):
    tx = generate_transaction(i)
    producer.send("transactions", value=tx)
    producer.flush()
    print(f"Sent: {tx}")
    time.sleep(1)

print("Finished sending 50 transactions.")

Writing producer.py


Part 3: 

Part 3.1: 

In [3]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='filter-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Listening for large transactions (amount > 1000)...")

for message in consumer:
    tx = message.value
    if tx["amount"] > 1000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']} PLN | {tx['store']} | {tx['category']}")

Writing consumer_filter.py


Part 3.2

In [4]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='enrich-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Listening and enriching transactions...")

for message in consumer:
    tx = message.value
    
    # Add risk level
    if tx["amount"] > 3000:
        tx["risk_level"] = "HIGH"
    elif tx["amount"] > 1000:
        tx["risk_level"] = "MEDIUM"
    else:
        tx["risk_level"] = "LOW"
    
    print(tx)

Writing consumer_enrich.py


Part 4: 

Part 4.1:

In [5]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0

print("Listening and counting transactions per store...")

for message in consumer:
    tx = message.value
    store = tx["store"]
    amount = tx["amount"]

    store_counts[store] += 1
    total_amount[store] += amount
    msg_count += 1

    if msg_count % 10 == 0:
        print("\n--- Store Summary ---")
        print(f"{'Store':<10} | {'Count':<5} | {'Total Amount':<12} | {'Avg Amount':<10}")
        print("-" * 55)

        for s in store_counts:
            count = store_counts[s]
            total = total_amount[s]
            avg = total / count if count > 0 else 0
            print(f"{s:<10} | {count:<5} | {total:<12.2f} | {avg:<10.2f}")

Writing consumer_count.py


Part 4.2:

In [8]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='stats-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = defaultdict(lambda: {
    "count": 0,
    "total_revenue": 0.0,
    "min_amount": float("inf"),
    "max_amount": float("-inf")
})

msg_count = 0

print("Listening and calculating category statistics...")

for message in consumer:
    tx = message.value
    category = tx["category"]
    amount = tx["amount"]

    stats[category]["count"] += 1
    stats[category]["total_revenue"] += amount
    stats[category]["min_amount"] = min(stats[category]["min_amount"], amount)
    stats[category]["max_amount"] = max(stats[category]["max_amount"], amount)

    msg_count += 1

    if msg_count % 10 == 0:
        print("\n--- Category Statistics ---")
        print(f"{'Category':<12} | {'Count':<5} | {'Revenue':<12} | {'Min':<10} | {'Max':<10}")
        print("-" * 65)

        for cat, data in stats.items():
            print(
                f"{cat:<12} | "
                f"{data['count']:<5} | "
                f"{data['total_revenue']:<12.2f} | "
                f"{data['min_amount']:<10.2f} | "
                f"{data['max_amount']:<10.2f}"
            )

Writing consumer_stats.py


Part 5.2:

Answer these questions:

1. What happens if you start consumer_filter.py AFTER the producer has finished?
If auto_offset_reset='earliest' and the consumer group has no committed offsets yet,
the consumer will read messages from the beginning, so it can still process old messages.

2. What happens if two consumers have the SAME group_id?
They join the same consumer group, so Kafka divides partitions/messages between them.
Each message will be processed by only one consumer in that group.

3. What is the difference between stateless and stateful processing?
Stateless processing handles each event independently without remembering previous events.
Example: consumer_filter.py checks each transaction amount and prints alerts.
Stateful processing keeps information across events, such as counters, totals, or statistics.
Example: consumer_count.py keeps running counts and totals per store.

Homework:

In [10]:
%%file consumer_velocity.py
from kafka import KafkaConsumer
from collections import defaultdict
from datetime import datetime, timedelta
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='velocity-group-1',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# Store timestamps for each user
user_timestamps = defaultdict(list)

print("Listening for velocity anomalies...")

for message in consumer:
    tx = message.value

    user_id = tx["user_id"]
    tx_id = tx["tx_id"]
    amount = tx["amount"]
    store = tx["store"]
    category = tx["category"]

    current_time = datetime.fromisoformat(tx["timestamp"])

    # Add current transaction time to the user's history
    user_timestamps[user_id].append(current_time)

    # Keep only timestamps from the last 60 seconds
    cutoff_time = current_time - timedelta(seconds=60)
    user_timestamps[user_id] = [
        t for t in user_timestamps[user_id] if t >= cutoff_time
    ]

    # Trigger alert if more than 3 transactions occurred in 60 seconds
    if len(user_timestamps[user_id]) > 3:
        print(
            f"VELOCITY ALERT: user {user_id} made "
            f"{len(user_timestamps[user_id])} transactions within 60 seconds | "
            f"Latest: {tx_id} | {amount:.2f} PLN | {store} | {category}"
        )

Writing consumer_velocity.py
